In [2]:
import sys
sys.path.append('../')

In [3]:
import yaml
from pathlib import Path
from project.evaluation import evaluate_on_benchmark_suite

In [5]:
CFG = "../BENCHMARK/configs/eval/whisper_baseline.yaml"
cfg = yaml.safe_load(Path(CFG).read_text())
rec = cfg["recognizer"]

# 어댑터 분기 (yaml 의 type)
if rec["type"] == "whisper":
    from project.data.adapters.whisper import build_predict_fn
    predict_fn = build_predict_fn(
        rec["model_path"],
        backbone=rec.get("backbone", "openai/whisper-small"),
        **rec.get("options", {}),
    )
elif rec["type"] == "sensevoice":
    from project.data.adapters.sensevoice import build_predict_fn
    predict_fn = build_predict_fn(rec["model_path"], **rec.get("options", {}))
else:
    raise ValueError(f"unknown recognizer.type: {rec['type']}")

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

In [9]:
# 벤치마크 ID → JSONL 경로 자동 lookup
benchmark_paths = {bid: f"../BENCHMARK/{bid}/samples.jsonl" for bid in cfg["benchmarks"]}

evaluate_on_benchmark_suite(
    model_name=rec["name"],
    predict_fn=predict_fn,
    benchmark_paths=benchmark_paths,
    out_dir=f"../BENCHMARK/results/{rec['name']}/",
    batch_size=cfg.get("batch_size", 16),
)

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take p

{'Sample10_PracticeRef': CerResult(cer=5.485232067510549, scer=3.825136612021858, wer=20.3125, samples=10, per_sample_cer=[1.9607843137254901, 8.695652173913043, 9.090909090909092, 57.14285714285714, 0.0, 3.0303030303030303, 2.380952380952381, 0.0, 4.166666666666666, 6.666666666666667])}